# Tertile sweep WITHOUT scVI (pseudobulk of raw counts)

Same analysis as the scVI tertile sweep of `Biomarker_prediction_scVI_pseudobulk_ML_h5ad_direct (11).ipynb`
(cells 36–72). The code is copied from that notebook. Only these things differ:

1. **No scVI model is loaded.** In `run_discovery_pdf_report`, the per-donor expression came from
   `model.get_normalized_expression(...)` averaged over cells. Here it comes from `pseudobulk_no_scvi()`
   applied to the raw counts (`layers["counts"]`). The library size is `obs["total_counts"]`, which covers the
   **full transcriptome**, not only the 2000 HVGs. Choose the transformation with `PSEUDOBULK_MODE`:
   * `"sum_logcpm"` (default): classic pseudobulk. Sum the counts over the donor's cells, divide by the donor's
     summed library, then log1p(CPM).
   * `"mean_cp10k"`: mean over cells of counts / total_counts × 1e4. This has the same scale as scVI's
     `library_size=1e4`, just without the denoising.
   * `"mean_lognorm"`: mean over cells of `layers["lognorm"]` = log1p(CP10k).
2. **`df_pb_latent` → `df_pb_clin`.** The scVI latent pseudobulk was only used for two things: to decide which
   donors and cell types enter the analysis (groups with ≥ 50 cells) and to carry the clinical columns. Here we
   use the same (donor × CellType, ≥ 50 cells) table, holding cell counts instead of latent dimensions.
3. **`cortis26np`** is not in the obs of this h5ad. It, and any other missing trait, is merged from the Dunedin
   phenotype file (`snum` = `patient_id`).
4. Outputs go to `MULTISPLIT_{BIOMARKER}_by_tertile_noscvi_{PSEUDOBULK_MODE}/`, with CSV and Excel files and save/resume.

Everything else is unchanged: labels, the outer 70/30 split, candidate genes from train cells, stability
selection, inner validation, the final XGBoost, and the AUCs.

In [ ]:
import os, re, gc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
from scipy import sparse
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display

In [ ]:
H5AD_PATH  = "/home/nlapique/Blood/adata_reduced_HVG_2000_with_sex_qcmt.h5ad"   # same file as the scVI run
PHENO_PATH = "/home/nlapique/Downloads/BroadInst_DunedinPheno_20240829.txt"      # for traits missing from obs

PSEUDOBULK_MODE = "sum_logcpm"   # <<< "sum_logcpm" | "mean_cp10k" | "mean_lognorm"

adata = sc.read_h5ad(H5AD_PATH)
print(adata)
assert "counts" in adata.layers and "total_counts" in adata.obs.columns

In [ ]:
# 'blood' = per-cell obs table, 'replicate' = patient key (same as the scVI notebook)
blood = adata.obs.copy()
blood["replicate"] = adata.obs["patient_id"].astype(str).str.strip().values

# traits used below that are not in this h5ad's obs -> merge them from the phenotype file
_needed = ["PaceOfAgingP45", "sex", "QC_MT.pct", "CurrentSmk26",
           "bmi26", "choles26", "trigly26", "hdl26", "urea26", "VOmax26", "MeanArtPr26",
           "LnCRP26", "crp26ns", "cortis26np", "PackYrsBy26"]
_missing = [c for c in _needed if c not in blood.columns]
if _missing:
    pheno = pd.read_csv(PHENO_PATH, sep="\t")
    pheno["replicate"] = pheno["snum"].astype(int).astype(str)
    _avail = [c for c in _missing if c in pheno.columns]
    blood = blood.join(pheno.set_index("replicate")[_avail], on="replicate")
    print(f"merged from phenotype file: {_avail}")
    if set(_missing) - set(_avail):
        print(f"⚠️ still missing: {sorted(set(_missing) - set(_avail))}")
print("blood:", blood.shape)

In [ ]:
def build_bio_indexed_from_blood(
    blood: pd.DataFrame,
    biomarker_cols: list = ["sex", "qt_mct"],
) -> pd.DataFrame:
    """
    Builds bio_indexed (patient_id × biomarkers) from blood (cells × variables),
    using the 'replicate' column as the patient identifier.
    """

    if "replicate" not in blood.columns:
        raise ValueError("The blood DataFrame must contain a 'replicate' column.")

    missing = [c for c in biomarker_cols if c not in blood.columns]
    if missing:
        raise ValueError(f"Missing biomarker columns in blood: {missing}")

    # 1) Keep only replicate + biomarkers
    tmp = blood[["replicate", *biomarker_cols]].copy()

    # 2) Drop rows where biomarkers are NaN (if desired)
    tmp = tmp.dropna(subset=biomarker_cols, how="any")

    # 3) Check there are no contradictions between cells of the same patient
    n_unique = tmp.groupby("replicate")[biomarker_cols].nunique()
    bad = n_unique[(n_unique > 1).any(axis=1)]
    if len(bad) > 0:
        print("⚠️ Patients with non-unique biomarkers in blood:")
        print(bad.head())
        print("→ We will still take the first value per patient.")
        
    # 4) Move to patient level: 1 row per patient
    bio_indexed = (
        tmp.drop_duplicates(subset="replicate")
           .set_index("replicate")
           .sort_index()
    )

    # 5) Rename the index to 'patient_id' to match df_latent_by_patient

    return bio_indexed


In [ ]:
# =====================================================================
# GLOBAL CONFIG  —  THE ONLY PLACE TO EDIT
# =====================================================================
BIOMARKER     = "PaceOfAgingP45"   # <<< THE ONLY PARAMETER TO CHANGE
TARGET_GROUP  = "All"                 # analyzed group (sex): "M" or "F"
N_ITER_PDF    = 50                  # iterations for the PDF report
N_ITER_EXCEL  = 50    
quantile_low=0.6
quantile_high=0.9
# iterations for the Excel gene export

# --- Automatic derived values (do not touch) ---
# Always-required covariates + the tested biomarker (deduplicated, order preserved)
EXTRA_GROUP_COLS = ["CurrentSmk26"]   # <<< extra covariates to carry through for alternative
                                      #     stratifications (e.g. splitting by smoking instead of sex)
BIO_COLS   = list(dict.fromkeys(["sex", "QC_MT.pct", BIOMARKER, *EXTRA_GROUP_COLS]))
EXCEL_FILE = f"RESULTATS_GENES_COMPLETS_{BIOMARKER}.xlsx"

# --- Check that the biomarker exists ---
try:
    if BIOMARKER not in blood.columns:
        cand = [c for c in blood.columns if BIOMARKER.lower() in c.lower()]
        raise KeyError(f"'{BIOMARKER}' not in blood.columns. Close matches: {cand}")
    print(f"✅ BIOMARKER = {BIOMARKER}")
    print(f"   BIO_COLS   = {BIO_COLS}")
    print(f"   EXCEL_FILE = {EXCEL_FILE}")
    print(f"   Groupe = {TARGET_GROUP} | n_iter PDF = {N_ITER_PDF} | n_iter Excel = {N_ITER_EXCEL}")
except NameError:
    print("⚠️ 'blood' not defined yet — run the .h5ad loading cell first.")


In [ ]:
bio_indexed = build_bio_indexed_from_blood(
    blood,
    biomarker_cols=BIO_COLS
)
bio_indexed

In [ ]:
def make_donor_celltype_table(
    adata,
    patient_col="patient_id",
    celltype_col="CellType",
    min_cells_per_group=50,
):
    """
    Replaces make_pseudobulk_latent_celltype: same (patient_id, CellType) groups with >= min_cells_per_group
    cells, but no scVI latent. Only the cell count is kept.
    index = MultiIndex (patient_id, CellType), column = n_cells
    """
    obs = adata.obs[[patient_col, celltype_col]].copy()
    obs[patient_col] = obs[patient_col].astype(str).str.strip()
    obs[celltype_col] = obs[celltype_col].astype(str)
    sizes = obs.groupby([patient_col, celltype_col]).size()
    df = sizes[sizes >= min_cells_per_group].rename("n_cells").to_frame().sort_index()
    print(f"{df.shape[0]} groups (patient x CellType) with >= {min_cells_per_group} cells, "
          f"{df.index.get_level_values(0).nunique()} patients")
    return df

df_counts_by_patient = make_donor_celltype_table(adata, min_cells_per_group=50)

df_pb_clin = df_counts_by_patient.join(
    bio_indexed,
    on="patient_id",   # join key
    how="inner"        # keep only common patients
)
df_pb_clin.head(1)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1. PARAMETERS ---
target_col = BIOMARKER   # Your continuous target
SEX_GROUP  = TARGET_GROUP       # "ALL" = males + females together | "M" / "F" for a single sex
                         #  -> should match target_group used in the XGBoost modeling

print(f"DISTRIBUTION ANALYSIS: {target_col}  (sex group: {SEX_GROUP})")

# --- 2. DATA PREPARATION ---
df_viz = df_pb_clin.copy()

# Clean up Index/IDs
if isinstance(df_viz.index, pd.MultiIndex): df_viz = df_viz.reset_index()
if 'patient_id' not in df_viz.columns:
    if 'index' in df_viz.columns: df_viz = df_viz.rename(columns={'index': 'patient_id'})
    else: df_viz['patient_id'] = df_viz.index.astype(str)

# Sex selection driven by SEX_GROUP (same mapping as the modeling functions)
sex_col = 'sex' if 'sex' in df_viz.columns else 'sex_patient'
df_viz[sex_col] = df_viz[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()
sex_map = {'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}
df_viz['sex_mapped'] = df_viz[sex_col].map(sex_map).fillna('Unknown')
if str(SEX_GROUP).upper() in ('ALL', 'BOTH'):
    print(f"   -> No sex filter (both sexes): {df_viz['patient_id'].nunique()} patients.")
else:
    df_viz = df_viz[df_viz['sex_mapped'] == SEX_GROUP]
    print(f"   -> Sex filter '{SEX_GROUP}': {df_viz['patient_id'].nunique()} patients remaining.")

# Remove duplicates (one patient = one value)
df_viz = df_viz.drop_duplicates(subset=['patient_id'])

# --- 3. THRESHOLD COMPUTATION ---
# Ignore missing values for the computation
values = df_viz[target_col].dropna()
low_thresh = values.quantile(quantile_low)
high_thresh = values.quantile(quantile_high)
print(f"   -> LOW threshold  : <= {low_thresh:.2f}  (q0.50)")
print(f"   -> HIGH threshold : >= {high_thresh:.2f}  (q0.90)")

# Build the labels for counting
def label_patient(val):
    if pd.isna(val): return "Missing"
    if val >= high_thresh: return "High (Sick)"
    elif val <= low_thresh: return "Low (Healthy)"
    else: return "Middle (Excluded)"

df_viz['Category'] = df_viz[target_col].map(label_patient)

# --- 4. VISUALIZATION ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PLOT 1: Continuous histogram (the shape of the data)
sns.histplot(df_viz[target_col], kde=True, ax=axes[0], color='grey', bins=30)
axes[0].axvline(low_thresh, color='blue', linestyle='--', linewidth=2, label=f'Low ({low_thresh:.1f})')
axes[0].axvline(high_thresh, color='red', linestyle='--', linewidth=2, label=f'High ({high_thresh:.1f})')

# Shade the zones
y_min, y_max = axes[0].get_ylim()
axes[0].fill_betweenx([y_min, y_max], df_viz[target_col].min(), low_thresh, color='blue', alpha=0.1)
axes[0].fill_betweenx([y_min, y_max], high_thresh, df_viz[target_col].max(), color='red', alpha=0.1)

axes[0].set_title(f"Continuous Distribution of {target_col}", fontsize=14)
axes[0].legend()
axes[0].set_xlabel("Biomarker Value")

# PLOT 2: Count bars (class balance)
order = ["Low (Healthy)", "Middle (Excluded)", "High (Sick)"]
palette = {"Low (Healthy)": "navy", "Middle (Excluded)": "lightgrey", "High (Sick)": "firebrick"}

sns.countplot(data=df_viz, x='Category', order=order, palette=palette, ax=axes[1])
axes[1].set_title("Number of Patients per Class", fontsize=14)

for container in axes[1].containers:
    axes[1].bar_label(container, fontsize=12, padding=3)

plt.tight_layout()
plt.show()

# --- 5. TEXT SUMMARY ---
print("\nHEADCOUNT SUMMARY:")
print(df_viz['Category'].value_counts().loc[order])


In [ ]:
# All cell types present in the data (instead of a fixed list).
# Source: the df_pb_latent pseudobulk (types actually available after aggregation),
# with fallback to adata.obs["CellType"].
try:
    my_valid_types = sorted(
        pd.Series(df_pb_clin.index.get_level_values("CellType")).astype(str).unique().tolist()
    )
    print("Source: df_pb_clin (CellType)")
except Exception:
    my_valid_types = sorted(adata.obs["CellType"].astype(str).unique().tolist())
    print("Source: adata.obs['CellType']")

print(f"{len(my_valid_types)} cell types:")
for t in my_valid_types:
    print("  -", t)


In [ ]:
def pseudobulk_no_scvi(adata_grp, gene_list, mode=None):
    """
    Donor x gene matrix from raw counts (replaces the scVI get_normalized_expression + mean).
    Library size = obs["total_counts"] (full transcriptome), as scVI/scanpy normalization would use.
    Returns a DataFrame indexed by patient_id (sorted, like groupby().mean()), columns = gene_list.
    """
    mode = mode or PSEUDOBULK_MODE
    idx = adata_grp.var_names.get_indexer(gene_list)
    if (idx < 0).any():
        raise KeyError("some genes of gene_list are not in var_names")
    pats = adata_grp.obs["patient_id"].astype(str).str.strip().values
    codes, uniq = pd.factorize(pats, sort=True)
    n = len(pats)
    G = sparse.csr_matrix((np.ones(n), (codes, np.arange(n))), shape=(len(uniq), n))   # donor x cell indicator
    n_cells = np.asarray(G.sum(axis=1)).ravel()
    lib = pd.to_numeric(adata_grp.obs["total_counts"], errors="coerce").to_numpy(dtype=float)

    if mode == "sum_logcpm":
        C = sparse.csr_matrix(adata_grp.layers["counts"][:, idx], dtype=np.float64)
        summed = np.asarray((G @ C).todense())
        out = np.log1p(summed / (G @ lib)[:, None] * 1e6)
    elif mode == "mean_cp10k":
        C = sparse.csr_matrix(adata_grp.layers["counts"][:, idx], dtype=np.float64)
        C = sparse.diags(1e4 / lib) @ C
        out = np.asarray((G @ C).todense()) / n_cells[:, None]
    elif mode == "mean_lognorm":
        L = sparse.csr_matrix(adata_grp.layers["lognorm"][:, idx], dtype=np.float64)
        out = np.asarray((G @ L).todense()) / n_cells[:, None]
    else:
        raise ValueError(f"unknown PSEUDOBULK_MODE {mode!r}")
    return pd.DataFrame(out.astype(np.float32), index=pd.Index(uniq, name="Patient"), columns=list(gene_list))

In [ ]:
def draw_table(df, title, pdf_object):
    fig, ax = plt.subplots(figsize=(8.5, 11)) # Standard A4 format
    ax.axis('off')
    
    # Page title
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Data preparation (rounding)
    table_data = df.copy()
    if 'Stability (%)' in table_data.columns:
        table_data['Stability (%)'] = table_data['Stability (%)'].apply(lambda x: f"{x:.1f}%")
    if 'Final Importance' in table_data.columns:
        table_data['Final Importance'] = table_data['Final Importance'].apply(lambda x: f"{x:.4f}")
    
    # Build the table
    cols = table_data.columns
    cell_text = table_data.values
    
    the_table = ax.table(
        cellText=cell_text,
        colLabels=cols,
        loc='center',
        cellLoc='center',
        colColours=['#eeeeee'] * len(cols) # Light grey header
    )
    
    # Styling
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(10)
    the_table.scale(1.2, 2) # More airy
    
    # Bold the header
    for (row, col), cell in the_table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold')
    
    pdf_object.savefig(fig, bbox_inches='tight')
    plt.close(fig)


In [ ]:
# --- MAIN FUNCTION (v2: train/test separation before gene selection + train-vs-test overfit gap) ---
def run_discovery_pdf_report(
    model, adata, df_clinical,
    biomarker="PaceOfAgingP45", cell_type="Classical_monocyte",
    n_iterations=50, stability_thresh=60, n_top_genes=1000, target_group="ALL",
    exclude_patients=None,        # <<< donors to remove this round (list of patient_id)
    make_pdf=True,                # <<< False -> skip PDF/SHAP/tree (fast, for the jackknife loop)
    min_auc=0.0,                  # <<< keep every round; set >0 only to drop weak runs
    outer_test_size=0.3,          # <<< NEW: held-out fraction, set aside BEFORE gene selection ever runs
    outer_random_state=42,        # <<< NEW: fixed split so repeated calls stay comparable
    min_train=15, min_test=5,     # <<< NEW: minimum group sizes on each side of the split
    n_perm=0,                     # <<< NEW: >0 runs a label-permutation null to calibrate stability_thresh
                                   #     (expensive: multiplies runtime by n_perm -> use standalone, NOT inside
                                   #      the jackknife loop in the next cell)
    n_iterations_perm=20,         # <<< NEW: iterations per permutation round (kept low to bound cost)
    inner_val_size=0.3,           # <<< NEW: carve this fraction OUT OF the train donors as a held-out
                                   #     inner-validation slice, scored with a fresh model using the
                                   #     already-selected stable_genes (outer test_pat stays untouched)
    n_estimators=500,             # <<< NEW: tree count for the FINAL and inner-val classifiers
                                   #     (the stability-selection XGB stays fixed at 300 trees / depth 3 --
                                   #      only the final model's capacity is exposed here for tuning)
    max_depth=4,                  # <<< NEW: max tree depth for the FINAL and inner-val classifiers
    group_col=None,               # <<< NEW: column to stratify on INSTEAD OF sex (e.g. "CurrentSmk26").
                                   #     None -> old behavior (stratify on sex_mapped, target_group="M"/"F"/"ALL").
                                   #     Set -> target_group is compared against THIS column's raw values
                                   #     (e.g. group_col="CurrentSmk26", target_group=1.0 for smokers).
    label_cutoffs="group",        # <<< NEW: where the High/Low biomarker cutoffs come from.
                                   #     "group"  -> quantiles of the target_group's donors only (old behavior:
                                   #                 each sex/tertile gets its OWN definition of High/Low)
                                   #     "global" -> quantiles of ALL donors in df_clinical, computed BEFORE the
                                   #                 target_group filter, so every group shares one definition
                                   #     (low, high) -> use these raw biomarker values directly as the cutoffs
):
    filename = f"Rapport_{biomarker}_{cell_type}_{target_group}.pdf"
    filename_csv = f"Tableau_{biomarker}_{cell_type}_{target_group}.csv"
    file_tree   = f"Arbre_{biomarker}_{cell_type}.pdf"
    if make_pdf: print(f"⚡️ ROBUST ANALYSIS & PDF REPORT: {filename}")

    # --- 1. LABELS (unchanged) ---
    pd.options.mode.chained_assignment = None
    df_clin = df_clinical.copy()
    if isinstance(df_clin.index, pd.MultiIndex): df_clin = df_clin.reset_index()
    if 'patient_id' not in df_clin.columns: df_clin['patient_id'] = df_clin.index.astype(str)
    df_clin['patient_id'] = df_clin['patient_id'].astype(str).str.strip()
    sex_col = 'sex' if 'sex' in df_clin.columns else 'sex_patient'
    df_clin[sex_col] = df_clin[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()

    sex_map = {'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}
    df_clin['sex_mapped'] = df_clin[sex_col].map(sex_map).fillna('Unknown')

    filter_col = group_col if group_col is not None else 'sex_mapped'
    if str(target_group).upper() in ('ALL', 'BOTH'):
        df_target = df_clin.drop_duplicates(subset=['patient_id'])
    else:
        df_target = df_clin[df_clin[filter_col] == target_group].drop_duplicates(subset=['patient_id'])

    if isinstance(label_cutoffs, (tuple, list)):
        low, high = label_cutoffs
    elif label_cutoffs == "global":
        df_all_donors = df_clin.drop_duplicates(subset=['patient_id'])
        high = df_all_donors[biomarker].quantile(quantile_high)
        low  = df_all_donors[biomarker].quantile(quantile_low)
    elif label_cutoffs == "group":
        high = df_target[biomarker].quantile(quantile_high)
        low  = df_target[biomarker].quantile(quantile_low)
    else:
        raise ValueError(f"label_cutoffs must be 'group', 'global' or a (low, high) tuple, got {label_cutoffs!r}")
    if low > high:
        raise ValueError(f"label cutoffs overlap: low={low} > high={high}")
    df_target['Label'] = np.nan
    df_target.loc[df_target[biomarker] >= high, 'Label'] = 1
    df_target.loc[df_target[biomarker] <= low, 'Label'] = 0
    df_ml = df_target.dropna(subset=['Label'])

    if exclude_patients is not None:
        excl = {str(x).strip() for x in exclude_patients}
        df_ml = df_ml[~df_ml['patient_id'].isin(excl)]

    if cell_type not in adata.obs['CellType'].values: return None
    if df_ml['Label'].nunique() < 2 or len(df_ml) < (min_train + min_test): return None

    if make_pdf:
        print(f"   -> Task: {len(df_ml[df_ml['Label']==1])} High vs {len(df_ml[df_ml['Label']==0])} Low "
              f"(High >= {high:.3f}, Low <= {low:.3f}, label_cutoffs={label_cutoffs!r})")

    # --- 2. OUTER SPLIT FIRST -- nothing below this line may look at the test patients ---
    #     until the single "evaluate the final model" step at the very end.
    train_pat, test_pat = train_test_split(
        df_ml['patient_id'], test_size=outer_test_size,
        random_state=outer_random_state, stratify=df_ml['Label']
    )
    train_pat, test_pat = set(train_pat), set(test_pat)
    labels_by_patient = df_ml.set_index('patient_id')['Label']

    adata_subset = adata[adata.obs['CellType'] == cell_type].copy()
    obs_pats = adata_subset.obs['patient_id'].astype(str).str.strip()
    adata_subset = adata_subset[obs_pats.isin(df_ml['patient_id'])].copy()
    subset_pats = adata_subset.obs['patient_id'].astype(str).str.strip()
    adata_train = adata_subset[subset_pats.isin(train_pat)].copy()
    adata_test  = adata_subset[subset_pats.isin(test_pat)].copy()

    def _pseudobulk(adata_grp, gene_list):
        # NO scVI: pseudobulk of raw counts (see pseudobulk_no_scvi / PSEUDOBULK_MODE); `model` is unused
        return pseudobulk_no_scvi(adata_grp, gene_list)

    # --- 3. CANDIDATE GENES from TRAIN cells only (unsupervised, but still test-blind) ---
    if hasattr(adata_train.X, "toarray"): means = np.array(adata_train.X.mean(axis=0)).flatten()
    else: means = np.mean(adata_train.X, axis=0)
    hvg = pd.Series(means, index=adata_train.var_names).sort_values(ascending=False).head(n_top_genes).index.tolist()

    X_train_full = _pseudobulk(adata_train, hvg)
    X_test_full  = _pseudobulk(adata_test, hvg)
    X_train_full = X_train_full.loc[X_train_full.index.intersection(train_pat)]
    X_test_full  = X_test_full.loc[X_test_full.index.intersection(test_pat)]
    y_train_full = labels_by_patient.loc[X_train_full.index]
    y_test_full  = labels_by_patient.loc[X_test_full.index]

    if len(X_train_full) < min_train or len(X_test_full) < min_test: return None
    if y_train_full.nunique() < 2 or y_test_full.nunique() < 2: return None

    def _stability_run(X, y, seed_offset):
        counts = pd.Series(0, index=hvg)
        for i in range(n_iterations):
            seed = seed_offset + i
            Xtr, _, ytr, _ = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
            clf = XGBClassifier(
                n_estimators=300, learning_rate=0.03, max_depth=3,
                subsample=0.8, colsample_bytree=0.6,
                device="cuda", tree_method="hist", eval_metric='logloss',   # <-- set device="cpu" if CUDA errors
                random_state=seed
            )
            clf.fit(Xtr, ytr)
            imps = pd.Series(clf.feature_importances_, index=X.columns)
            counts[imps[imps > 0].index] += 1
        return counts

    # --- 4. ROBUSTNESS / STABILITY -- resampled ONLY inside the TRAIN patients ---
    if make_pdf: print(f"   -> Computing robustness on TRAIN only ({n_iterations} runs)", end="")
    gene_counts = _stability_run(X_train_full, y_train_full, seed_offset=42)
    if make_pdf: print(" Done.")
    stability_pct = (gene_counts / n_iterations) * 100

    # --- 5. OPTIONAL: label-permutation null, also computed on TRAIN only ---
    effective_thresh = stability_thresh
    empirical_thresh = None
    if n_perm > 0:
        if make_pdf: print(f"   -> Calibrating threshold with {n_perm} label permutations", end="")
        rng = np.random.default_rng(0)
        null_max = []
        for p in range(n_perm):
            y_perm = pd.Series(rng.permutation(y_train_full.values), index=y_train_full.index)
            perm_counts = pd.Series(0, index=hvg)
            for i in range(n_iterations_perm):
                seed = 900_000 + p * 1000 + i
                Xtr, _, ytr, _ = train_test_split(X_train_full, y_perm, test_size=0.3, random_state=seed, stratify=y_perm)
                clf = XGBClassifier(
                    n_estimators=300, learning_rate=0.03, max_depth=3,
                    subsample=0.8, colsample_bytree=0.6,
                    device="cuda", tree_method="hist", eval_metric='logloss',
                    random_state=seed
                )
                clf.fit(Xtr, ytr)
                imps = pd.Series(clf.feature_importances_, index=X_train_full.columns)
                perm_counts[imps[imps > 0].index] += 1
            null_max.append((perm_counts / n_iterations_perm * 100).max())
            if make_pdf and p % 5 == 0: print(".", end="")
        if make_pdf: print(" Done.")
        empirical_thresh = float(np.percentile(null_max, 95))
        effective_thresh = max(stability_thresh, empirical_thresh)
        if make_pdf:
            print(f"   -> Null 95th percentile stability: {empirical_thresh:.1f}% "
                  f"(using max({stability_thresh}, {empirical_thresh:.1f}) = {effective_thresh:.1f}% as cutoff)")

    stable_genes = stability_pct[stability_pct >= effective_thresh].index.tolist()
    if len(stable_genes) < 2:
        stable_genes = stability_pct.sort_values(ascending=False).head(10).index.tolist()

    X_train_stable = X_train_full[stable_genes]
    X_test_stable  = X_test_full[stable_genes]

    # --- 5b. INNER VALIDATION -- split the TRAIN donors again (sub-train / sub-val), fit a FRESH
    #     model on the sub-train slice using the stable_genes already chosen above, and score it on
    #     the sub-val slice. This never touches test_pat; it's a second, independent check of
    #     generalization using only donors the outer split already called "train".
    inner_val_auc = np.nan
    n_subtrain = n_subval = 0
    if y_train_full.nunique() == 2 and len(X_train_stable) >= (min_train + min_test):
        try:
            sub_train_pat, sub_val_pat = train_test_split(
                X_train_stable.index, test_size=inner_val_size,
                random_state=outer_random_state, stratify=y_train_full.loc[X_train_stable.index]
            )
            X_subtrain, X_subval = X_train_stable.loc[sub_train_pat], X_train_stable.loc[sub_val_pat]
            y_subtrain, y_subval = y_train_full.loc[sub_train_pat], y_train_full.loc[sub_val_pat]
            n_subtrain, n_subval = len(X_subtrain), len(X_subval)
            if y_subtrain.nunique() == 2 and y_subval.nunique() == 2:
                inner_clf = XGBClassifier(
                    n_estimators=n_estimators, learning_rate=0.02, max_depth=max_depth,
                    subsample=0.8, colsample_bytree=0.8,
                    device="cuda", tree_method="hist", eval_metric='logloss',   # <-- set device="cpu" if CUDA errors
                    random_state=42
                )
                inner_clf.fit(X_subtrain, y_subtrain)
                y_proba_subval = inner_clf.predict_proba(X_subval)[:, 1]
                fpr_iv, tpr_iv, _ = roc_curve(y_subval, y_proba_subval)
                inner_val_auc = auc(fpr_iv, tpr_iv)
        except ValueError:
            pass   # too few donors of one class to stratify the inner split -> leave as NaN

    # --- 6. FINAL MODEL -- fit on ALL train patients, evaluate on the never-touched test patients ---
    final_clf = XGBClassifier(
        n_estimators=n_estimators, learning_rate=0.02, max_depth=max_depth,
        subsample=0.8, colsample_bytree=0.8,
        device="cuda", tree_method="hist", eval_metric='logloss',       # <-- set device="cpu" if CUDA errors
        random_state=42
    )
    final_clf.fit(X_train_stable, y_train_full)

    y_proba = final_clf.predict_proba(X_test_stable)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_full, y_proba)
    roc_auc = auc(fpr, tpr)

    # --- overfitting metric: same model, scored on the TRAIN patients it was fit on ---
    y_proba_train = final_clf.predict_proba(X_train_stable)[:, 1]
    fpr_tr, tpr_tr, _ = roc_curve(y_train_full, y_proba_train)
    train_auc = auc(fpr_tr, tpr_tr)
    overfit_gap = train_auc - roc_auc   # large positive gap -> memorizing train, not generalizing

    # gap between in-sample train score and the FRESH inner-val model's score, both computed without
    # ever touching test_pat -- cheaper/noisier than overfit_gap (which uses the real held-out test
    # set) but usable for hyperparameter tuning without peeking at test.
    overfit_gap_innerval = (train_auc - inner_val_auc) if not np.isnan(inner_val_auc) else np.nan

    if make_pdf:
        print(f"🎯 TEST (held-out) AUC: {roc_auc:.2f} | TRAIN AUC: {train_auc:.2f} | "
              f"OVERFIT GAP: {overfit_gap:+.2f}")
        print(f"   -> Inner-val AUC (fresh model, sub-train {n_subtrain} / sub-val {n_subval}, "
              f"carved out of the 70% train donors, real test set untouched): {inner_val_auc:.2f} "
              f"| INNER OVERFIT GAP: {overfit_gap_innerval:+.2f}")
    if roc_auc < min_auc:
        return None

    # per-patient scores: TEST scores are honest (model never saw these patients before this line);
    # TRAIN scores are in-sample/optimistic and flagged as such.
    all_ids = list(X_train_stable.index) + list(X_test_stable.index)
    all_X = pd.concat([X_train_stable, X_test_stable])
    scores_all = final_clf.predict_proba(all_X)[:, 1]
    lbl = labels_by_patient.loc[all_ids].astype(int)
    df_scores = pd.DataFrame({
        "patient_id": all_ids,
        "label": lbl.values,
        "category": np.where(lbl.values == 1, "High", "Low"),
        "score_high": scores_all,
        "predicted": np.where(scores_all >= 0.5, "High", "Low"),
        "in_train": [pid in X_train_stable.index for pid in all_ids],  # True = optimistic (seen in training)
    })

    # --- 7. TABLE PREPARATION ---
    imp_map = dict(zip(stable_genes, final_clf.feature_importances_))
    df_all = pd.DataFrame({"Gene": stability_pct.index, "Stability (%)": stability_pct.values})
    df_all = df_all[df_all["Stability (%)"] > 50].sort_values("Stability (%)", ascending=False)
    df_all["Final Importance"] = df_all["Gene"].map(imp_map)

    df_res = pd.DataFrame({
        "Gene": stable_genes,
        "Stability (%)": stability_pct[stable_genes].values,
        "Final Importance": final_clf.feature_importances_
    }).sort_values("Stability (%)", ascending=False).head(25)

    # --- 8. PDF (only on demand) ---
    if make_pdf:
        df_all.to_csv(filename_csv, index=False, sep=';', decimal=',')
        print(f"   -> 💾 CSV saved ({len(df_all)} genes >50%): {filename_csv}")
        print(f"   -> Creating PDF file: {filename} ...")
        with PdfPages(filename) as pdf:
            fig_roc = plt.figure(figsize=(8, 6))
            plt.plot(fpr, tpr, color='#d62728', lw=3, label=f'Test ROC (AUC = {roc_auc:.2f}, held-out)')
            plt.plot(fpr_tr, tpr_tr, color='#1f77b4', lw=2, linestyle=':',
                     label=f'Train ROC (AUC = {train_auc:.2f}, in-sample)')
            plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
            plt.title(f"Diagnostic Performance: {biomarker} ({cell_type})\nOverfit gap = {overfit_gap:+.2f}", fontsize=14)
            plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
            plt.legend(loc="lower right"); plt.grid(alpha=0.3)
            pdf.savefig(fig_roc); plt.show(); plt.close(fig_roc)

            explainer = shap.TreeExplainer(final_clf)
            shap_values = explainer.shap_values(X_train_stable)  # SHAP on train is fine, it's just explaining the model
            fig_shap = plt.figure()
            shap.summary_plot(shap_values, X_train_stable, show=False, max_display=20)
            plt.title(f"Biological Signature (SHAP) - {cell_type}", fontsize=14)
            plt.tight_layout(); pdf.savefig(fig_shap, bbox_inches='tight'); plt.show(); plt.close(fig_shap)

            draw_table(df_res, f"Top Robust Genes: {cell_type}", pdf)
            try:
                fig_tree, ax_tree = plt.subplots(figsize=(30, 40))
                xgb.plot_tree(final_clf, num_trees=0, ax=ax_tree, rankdir='TB')
                plt.title(f"Vertical Decision Tree - {cell_type}", fontsize=30, pad=30)
                plt.savefig(file_tree, bbox_inches='tight'); plt.close(fig_tree)
                print(f"   -> 🌳 Vertical tree saved: {file_tree}")
            except Exception as e:
                print(f"   ⚠️ Tree error: {e}")
        print(f"✅ Done! File saved: {filename}")

    return {"roc_auc": roc_auc, "train_auc": train_auc, "overfit_gap": overfit_gap,
            "inner_val_auc": inner_val_auc, "overfit_gap_innerval": overfit_gap_innerval,
            "n_subtrain": n_subtrain, "n_subval": n_subval,
            "df_all": df_all, "df_res": df_res,
            "final_clf": final_clf, "donors": list(train_pat | test_pat),
            "df_scores": df_scores,
            "empirical_stability_threshold": empirical_thresh,
            "stability_threshold_used": effective_thresh,
            "outer_random_state": outer_random_state, "n_train": len(X_train_full), "n_test": len(X_test_full),
            "label_cutoff_low": low, "label_cutoff_high": high}


In [ ]:
# --- MULTI-SPLIT WRAPPER: repeats the outer train/test split to get a DISTRIBUTION of held-out AUCs ---
# A single 70/30 split on ~dozens of patients is a high-variance estimate. This reruns
# run_discovery_pdf_report() with several independent outer_random_state seeds and reports
# mean/std of test AUC, train AUC and the overfit gap, plus how often each gene is picked
# as "stable" across splits (a gene that's only stable in 1 of 10 splits is not robust).
def run_discovery_multi_split(
    model, adata, df_clinical,
    biomarker="PaceOfAgingP45", cell_type="Classical_monocyte",
    n_iterations=50, stability_thresh=60, n_top_genes=1000, target_group="ALL",
    exclude_patients=None,
    n_outer_splits=10,             # number of independent outer train/test splits
    outer_split_seeds=None,        # optional explicit seed list; default = range(n_outer_splits)
    outer_test_size=0.3,
    min_train=15, min_test=5,
    n_perm=0, n_iterations_perm=20,
    inner_val_size=0.3,            # passed through to run_discovery_pdf_report's inner validation split
    n_estimators=500,              # passed through -- tree count for the final/inner-val classifiers
    max_depth=4,                   # passed through -- max tree depth for the final/inner-val classifiers
    group_col=None,                # passed through -- stratify on this column instead of sex (see
                                    # run_discovery_pdf_report's docstring-comment for group_col)
    label_cutoffs="group",         # passed through -- "group" (old), "global", or a (low, high) tuple
                                    # (see run_discovery_pdf_report's comment for label_cutoffs)
    make_pdf_for_first=False,      # generate the full PDF/SHAP/tree report only for the first split
):
    seeds = list(outer_split_seeds) if outer_split_seeds is not None else list(range(n_outer_splits))
    rows = []
    gene_hits = {}   # gene -> number of splits where it landed in the top-25 "stable" table

    for k, seed in enumerate(seeds):
        res = run_discovery_pdf_report(
            model=model, adata=adata, df_clinical=df_clinical,
            biomarker=biomarker, cell_type=cell_type,
            n_iterations=n_iterations, stability_thresh=stability_thresh,
            n_top_genes=n_top_genes, target_group=target_group,
            exclude_patients=exclude_patients,
            make_pdf=(make_pdf_for_first and k == 0),
            min_auc=0.0,
            outer_test_size=outer_test_size,
            outer_random_state=seed,
            min_train=min_train, min_test=min_test,
            n_perm=n_perm, n_iterations_perm=n_iterations_perm,
            inner_val_size=inner_val_size,
            n_estimators=n_estimators, max_depth=max_depth,
            group_col=group_col,
            label_cutoffs=label_cutoffs,
        )
        if res is None:
            rows.append({"split": k, "seed": seed, "test_auc": np.nan, "train_auc": np.nan,
                         "inner_val_auc": np.nan, "overfit_gap": np.nan, "overfit_gap_innerval": np.nan,
                         "n_train": np.nan, "n_test": np.nan, "n_stable_genes": np.nan})
            continue
        rows.append({
            "split": k, "seed": seed,
            "test_auc": res["roc_auc"], "train_auc": res["train_auc"],
            "inner_val_auc": res["inner_val_auc"], "overfit_gap": res["overfit_gap"],
            "overfit_gap_innerval": res["overfit_gap_innerval"],
            "n_train": res["n_train"], "n_test": res["n_test"], "n_stable_genes": len(res["df_res"]),
        })
        for g in res["df_res"]["Gene"]:
            gene_hits[g] = gene_hits.get(g, 0) + 1

    df_splits = pd.DataFrame(rows)
    n_valid = int(df_splits["test_auc"].notna().sum())

    summary = {
        "cell_type": cell_type, "biomarker": biomarker, "target_group": target_group,
        "label_cutoffs": label_cutoffs,
        "n_splits_requested": len(seeds), "n_splits_valid": n_valid,
        "test_auc_mean": df_splits["test_auc"].mean(), "test_auc_std": df_splits["test_auc"].std(),
        "train_auc_mean": df_splits["train_auc"].mean(),
        "inner_val_auc_mean": df_splits["inner_val_auc"].mean(), "inner_val_auc_std": df_splits["inner_val_auc"].std(),
        "overfit_gap_mean": df_splits["overfit_gap"].mean(), "overfit_gap_std": df_splits["overfit_gap"].std(),
        "overfit_gap_innerval_mean": df_splits["overfit_gap_innerval"].mean(),
        "overfit_gap_innerval_std": df_splits["overfit_gap_innerval"].std(),
    }

    print(f"📊 {cell_type} | {biomarker} ({target_group}) over {n_valid}/{len(seeds)} valid splits:")
    print(f"   Test AUC       : {summary['test_auc_mean']:.3f} ± {summary['test_auc_std']:.3f}")
    print(f"   Inner-val AUC  : {summary['inner_val_auc_mean']:.3f} ± {summary['inner_val_auc_std']:.3f}  "
          f"(fresh model, held out of the 70% train donors)")
    print(f"   Train AUC      : {summary['train_auc_mean']:.3f}")
    print(f"   Overfit gap    : {summary['overfit_gap_mean']:+.3f} ± {summary['overfit_gap_std']:.3f}  "
          f"(train AUC minus TEST AUC; large positive = memorizing, not generalizing)")
    print(f"   Inner ovft gap : {summary['overfit_gap_innerval_mean']:+.3f} ± {summary['overfit_gap_innerval_std']:.3f}  "
          f"(train AUC minus INNER-VAL AUC)")

    genes_across_splits = (
        pd.Series(gene_hits, dtype=float)
          .sort_values(ascending=False)
          .rename("times_in_top25")
          .to_frame()
    )
    if len(seeds) > 0:
        genes_across_splits["pct_of_splits"] = genes_across_splits["times_in_top25"] / len(seeds) * 100

    return {"df_splits": df_splits, "summary": summary, "genes_across_splits": genes_across_splits}


In [ ]:
# --- Low/Medium/High tertile categories for age-26 continuous traits ---
# Pulled straight from `blood` (not through BIO_COLS) so each trait keeps its own ~89-96%
# completeness instead of all 9 traits being required non-null simultaneously (that would
# drop the cohort from 807 to 698 donors -- checked before doing this).
TRAITS_FOR_TERTILE = ["crp26ns","hdl26","bmi26", "choles26", "trigly26",  "urea26",
                      "VOmax26", "MeanArtPr26", "LnCRP26", "cortis26np","PackYrsBy26"]
#TRAITS_FOR_TERTILE = ["hdl26"]

trait_indexed = (
    blood[["replicate", *TRAITS_FOR_TERTILE]]
    .drop_duplicates(subset="replicate")
    .set_index("replicate")
)

for t in TRAITS_FOR_TERTILE:
    cat_col = f"{t}_cat"
    try:
        trait_indexed[cat_col] = pd.qcut(
            trait_indexed[t], q=3, labels=["Low", "Medium", "High"], duplicates="drop"
        )
    except ValueError as e:
        # Tied mass at the minimum (e.g. PackYrsBy26 = 0 for all non-smokers) collapses the tertile
        # edges. Fallback: Low = donors at the minimum value, Medium/High = median split of the rest.
        s = trait_indexed[t]
        at_min = s == s.min()
        rest = s[s.notna() & ~at_min]
        if rest.nunique() < 2:
            print(f"⚠️ Could not tertile-split {t}: {e}")
            continue
        cat = pd.Series(pd.NA, index=s.index, dtype=object)
        cat[at_min] = "Low"
        cat[rest.index] = pd.qcut(rest, q=2, labels=["Medium", "High"]).astype(object)
        trait_indexed[cat_col] = pd.Categorical(cat, categories=["Low", "Medium", "High"])
        print(f"   {t}: ties at min={s.min()} -> Low = at min, Medium/High = median split of the rest")
    vc = trait_indexed[cat_col].value_counts(dropna=False)
    n_non_null = trait_indexed[t].notna().sum()
    print(f"{t}: {n_non_null} donors with a value -> {dict(vc)}")

cat_cols = [f"{t}_cat" for t in TRAITS_FOR_TERTILE if f"{t}_cat" in trait_indexed.columns]
df_pb_clin = (df_pb_clin.drop(columns=cat_cols, errors="ignore")   # safe to re-run
                .join(trait_indexed[cat_cols], on="patient_id", how="left"))
df_pb_clin[cat_cols].notna().sum()


In [ ]:
import os, re, gc
# --- USAGE: ALL cell types, split by Low/Medium/High tertile of each age-26 trait ---
# Same run_discovery_multi_split(group_col=...) mechanism as the smoking sweep above, but looped
# over traits x 3 tertile levels x cell types. This is BOTH sexes pooled within each tertile
# (group_col points at the trait's *_cat column, not sex_mapped).
#
# PROGRESSIVE SAVE + RESUME: after every (cell_type, trait, tertile) run, the summary CSV is rewritten
# and that run's per-split AUCs and gene hits are written to OUTDIR_TERTILE/per_combo/. If the kernel
# crashes, just re-run this cell: combos already in the summary CSV are loaded from disk and skipped.
# Set RESUME_TERTILE = False to start from scratch (the old files are then overwritten as runs finish).
# Crashed combos are logged to crashes_by_tertile.csv and are retried on the next run.
#
# Cost note: len(TRAITS_FOR_TERTILE) x 3 levels x len(CELL_TYPES_FOR_TERTILE_SWEEP)
# x N_OUTER_SPLITS_TERTILE full pipeline runs.
_missing_cat = [t for t in TRAITS_FOR_TERTILE if f"{t}_cat" not in df_pb_clin.columns]
assert len(_missing_cat) < len(TRAITS_FOR_TERTILE), (
    "No tertile category columns in df_pb_clin -- run the tertile-category cell above first."
)
if _missing_cat:
    print(f"⚠️ No _cat column for {_missing_cat} -- these traits are skipped in the sweep.")

CELL_TYPES_FOR_TERTILE_SWEEP = my_valid_types   # <<< EDIT: shrink this list to bound runtime, e.g.
                                                 #     ["CD4 Active", "B Naive", "B Memory"]
N_OUTER_SPLITS_TERTILE = 20
TERTILE_LEVELS = ["Low", "Medium", "High"]
RESUME_TERTILE = True                            # <<< False = ignore previous results and redo everything

OUTDIR_TERTILE = f"MULTISPLIT2_{BIOMARKER}_by_tertile_noscvi_{PSEUDOBULK_MODE}"
COMBO_DIR_TERTILE = f"{OUTDIR_TERTILE}/per_combo"
SUMMARY_TERTILE_CSV = f"{OUTDIR_TERTILE}/summary_all_celltypes_by_tertile_noscvi.csv"
CRASH_TERTILE_CSV = f"{OUTDIR_TERTILE}/crashes_by_tertile.csv"
os.makedirs(COMBO_DIR_TERTILE, exist_ok=True)

def _combo_tag(cell_type, trait, level):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{trait}__{level}__{cell_type}")

XLSX_TERTILE = f"{OUTDIR_TERTILE}/RESULTATS_TERTILE_NOSCVI_{PSEUDOBULK_MODE}_{BIOMARKER}.xlsx"

def _write_excel_tertile(summary_rows, results, crash_rows, path):
    """Rebuild the whole workbook from memory; written to a temp file, then swapped in."""
    first = ["cell_type", "trait", "tertile"]
    splits, genes = [], []
    for (ct, trait, lv), r in results.items():
        if r.get("df_splits") is not None:
            splits.append(r["df_splits"].assign(cell_type=ct, trait=trait, tertile=lv))
        g = r.get("genes_across_splits")
        if g is not None and len(g):
            g = g.reset_index()
            g = g.rename(columns={g.columns[0]: "Gene"})
            genes.append(g.assign(cell_type=ct, trait=trait, tertile=lv))
    df_sum = pd.DataFrame(summary_rows)
    if len(df_sum):
        df_sum = df_sum.sort_values("test_auc_mean", ascending=False)
    df_spl = pd.concat(splits, ignore_index=True) if splits else pd.DataFrame(columns=first)
    df_gen = pd.concat(genes, ignore_index=True) if genes else pd.DataFrame(columns=first)
    df_spl = df_spl[first + [c for c in df_spl.columns if c not in first]]
    df_gen = df_gen[first + [c for c in df_gen.columns if c not in first]]
    settings = pd.DataFrame({
        "setting": ["biomarker", "features", "pseudobulk_mode", "n_outer_splits", "n_iterations",
                    "quantile_low", "quantile_high", "cell_types", "traits"],
        "value": [BIOMARKER, "pseudobulk of raw counts, library = total_counts (full transcriptome), no scVI",
                  PSEUDOBULK_MODE, N_OUTER_SPLITS_TERTILE, N_ITER_PDF, quantile_low, quantile_high,
                  ", ".join(map(str, CELL_TYPES_FOR_TERTILE_SWEEP)), ", ".join(TRAITS_FOR_TERTILE)]})
    tmp = path[:-5] + ".tmp.xlsx"
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        df_sum.to_excel(xw, sheet_name="summary", index=False)
        df_spl.to_excel(xw, sheet_name="splits", index=False)
        df_gen.to_excel(xw, sheet_name="genes", index=False)
        pd.DataFrame(crash_rows, columns=["cell_type", "trait", "tertile", "error"]).to_excel(
            xw, sheet_name="crashes", index=False)
        settings.to_excel(xw, sheet_name="settings", index=False)
    os.replace(tmp, path)

def _write_csv_atomic(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False, sep=';', decimal=',')
    os.replace(tmp, path)   # never leaves a half-written file if the kernel dies mid-write

# --- resume: load what previous runs already saved ---
all_rows_tertile = []
per_celltype_results_tertile = {}   # (cell_type, trait, level) -> {"df_splits", "summary", "genes_across_splits"}
done_tertile = set()
if RESUME_TERTILE and os.path.exists(SUMMARY_TERTILE_CSV):
    _prev = pd.read_csv(SUMMARY_TERTILE_CSV, sep=';', decimal=',')
    # only resume combos run with the same number of splits
    _prev = _prev[_prev["n_splits_requested"] == N_OUTER_SPLITS_TERTILE]
    all_rows_tertile = _prev.to_dict("records")
    for r in all_rows_tertile:
        key = (r["cell_type"], r["trait"], r["tertile"])
        done_tertile.add(key)
        tag = _combo_tag(*key)
        _p_splits = f"{COMBO_DIR_TERTILE}/{tag}__splits.csv"
        _p_genes = f"{COMBO_DIR_TERTILE}/{tag}__genes.csv"
        per_celltype_results_tertile[key] = {
            "summary": r,
            "df_splits": pd.read_csv(_p_splits, sep=';', decimal=',') if os.path.exists(_p_splits) else None,
            "genes_across_splits": (pd.read_csv(_p_genes, sep=';', decimal=',', index_col=0)
                                    if os.path.exists(_p_genes) else None),
        }
    print(f"↩️ Resuming: {len(done_tertile)} combos already done (loaded from {SUMMARY_TERTILE_CSV})")

crash_rows_tertile = []
_todo = [(t, ct, lv) for t in TRAITS_FOR_TERTILE if f"{t}_cat" in df_pb_clin.columns
         for ct in CELL_TYPES_FOR_TERTILE_SWEEP for lv in TERTILE_LEVELS
         if (ct, t, lv) not in done_tertile]
print(f"▶️ {len(_todo)} combos to run")

for i_combo, (trait, cell_type, level) in enumerate(_todo, 1):
    cat_col = f"{trait}_cat"
    print(f"\n===== [{i_combo}/{len(_todo)}] {cell_type} | {trait} = {level} =====")
    try:
        res = run_discovery_multi_split(
            model=None, adata=adata, df_clinical=df_pb_clin,
            biomarker=BIOMARKER, cell_type=cell_type,
            n_iterations=N_ITER_PDF, target_group=level,
            group_col=cat_col,
            n_outer_splits=N_OUTER_SPLITS_TERTILE,
        )
    except Exception as e:
        print(f"⚠️ CRASH on {cell_type} ({trait}={level}): {e}")
        crash_rows_tertile.append({"cell_type": cell_type, "trait": trait, "tertile": level,
                                   "error": repr(e)})
        _write_csv_atomic(pd.DataFrame(crash_rows_tertile), CRASH_TERTILE_CSV)
        continue

    key = (cell_type, trait, level)
    per_celltype_results_tertile[key] = res
    s = res["summary"]
    all_rows_tertile.append({
        "cell_type": cell_type, "trait": trait, "tertile": level,
        "n_splits_valid": s["n_splits_valid"], "n_splits_requested": s["n_splits_requested"],
        "test_auc_mean": s["test_auc_mean"], "test_auc_std": s["test_auc_std"],
        "inner_val_auc_mean": s["inner_val_auc_mean"], "inner_val_auc_std": s["inner_val_auc_std"],
        "train_auc_mean": s["train_auc_mean"],
        "overfit_gap_mean": s["overfit_gap_mean"], "overfit_gap_std": s["overfit_gap_std"],
        "overfit_gap_innerval_mean": s["overfit_gap_innerval_mean"],
        "overfit_gap_innerval_std": s["overfit_gap_innerval_std"],
    })

    # --- progressive save: this combo's details + the full summary so far ---
    tag = _combo_tag(*key)
    _write_csv_atomic(res["df_splits"], f"{COMBO_DIR_TERTILE}/{tag}__splits.csv")
    _g = res["genes_across_splits"].reset_index().rename(columns={"index": "Gene"})
    _write_csv_atomic(_g, f"{COMBO_DIR_TERTILE}/{tag}__genes.csv")
    _write_csv_atomic(pd.DataFrame(all_rows_tertile).sort_values("test_auc_mean", ascending=False),
                      SUMMARY_TERTILE_CSV)
    try:
        _write_excel_tertile(all_rows_tertile, per_celltype_results_tertile, crash_rows_tertile, XLSX_TERTILE)
    except Exception as e:   # CSVs are the resume source; an Excel problem must not stop the sweep
        print(f"⚠️ Excel write failed ({e!r})")
    print(f"💾 saved ({len(all_rows_tertile)} combos in {SUMMARY_TERTILE_CSV} + .xlsx)")
    gc.collect()

_write_excel_tertile(all_rows_tertile, per_celltype_results_tertile, crash_rows_tertile, XLSX_TERTILE)
print(f"📗 Excel: {os.path.abspath(XLSX_TERTILE)}")
df_all_celltypes_tertile = pd.DataFrame(all_rows_tertile)
if len(df_all_celltypes_tertile):
    df_all_celltypes_tertile = df_all_celltypes_tertile.sort_values("test_auc_mean", ascending=False)
if crash_rows_tertile:
    print(f"⚠️ {len(crash_rows_tertile)} combos crashed -- see {CRASH_TERTILE_CSV}; re-run this cell to retry them.")
df_all_celltypes_tertile